In [1]:
!git clone https://github.com/dhakalnirajan/internship-projects.git
%cd internship-projects/pothole-detection

# Install requirements
!pip install -r requirements.txt -q

Cloning into 'internship-projects'...
remote: Enumerating objects: 275, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 275 (delta 9), reused 58 (delta 5), pack-reused 205 (from 1)
Receiving objects: 100% (275/275), 52.35 MiB | 27.38 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/internship-projects/pothole-detection
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 9.6 MB/s eta 0:00:00


In [2]:
# Configure Kaggle API credentials (modify username and key if needed, or prompt for input)
import os
import json
from google.colab import userdata

os.makedirs('/root/.kaggle', exist_ok=True)
try:
    kaggle_creds = userdata.get('Kaggle')
    # If the secret is stored as a JSON string or file contents
    if isinstance(kaggle_creds, str):
        try:
            creds_dict = json.loads(kaggle_creds)
        except json.JSONDecodeError:
            # Assume it might just be the key or handle accordingly; if it's the full JSON string
            raise ValueError("Kaggle secret is not valid JSON string.")
    else:
        creds_dict = kaggle_creds

    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(creds_dict, f)

    os.chmod('/root/.kaggle/kaggle.json', 600)
    print("Kaggle credentials successfully loaded from Colab Secret 'Kaggle'.")
except Exception as e:
    print(f"Error loading credentials: {e}. Please check your Colab Secret name.")

Error loading credentials: Kaggle secret is not valid JSON string.. Please check your Colab Secret name.


In [3]:
# Download and extract dataset to the designated directory
!kaggle datasets download -d andrewmvd/pothole-detection
!rm data
!unzip -q pothole-detection.zip -d data
!rm pothole-detection.zip

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/pothole-detection
License(s): other
100% 334M/334M [00:02<00:00, 130MB/s]

rm: cannot remove 'data': Is a directory


In [4]:
# Verify dataset structure and adjust config.yaml if necessary
import yaml
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

print("Current config.yaml content:", config)

Current config.yaml content: {'data': {'raw_images': 'data/raw/images', 'raw_annotations': 'data/raw/annotations', 'processed_images': 'data/processed/images', 'processed_labels': 'data/processed/labels', 'dataset_yaml': 'data/dataset.yaml'}, 'train': {'model_size': 'yolov8s.pt', 'epochs': 100, 'batch_size': 16, 'img_size': 640, 'lr0': 0.01, 'lrf': 0.01, 'weight_decay': 0.0005, 'augment': True, 'patience': 20}, 'inference': {'conf_threshold': 0.6, 'iou_threshold': 0.5, 'frame_skip': 3, 'max_age': 30, 'min_hits': 3, 'alert_distance_meters': 5.0}, 'export': {'format': 'onnx', 'half': True, 'int8': False}, 'logging': {'level': 'INFO', 'file': 'pothole.log'}}


In [5]:
# Create required directories
os.makedirs(config['data']['processed_images'], exist_ok=True)
os.makedirs(os.path.dirname(config['data']['dataset_yaml']), exist_ok=True)

In [6]:
# Populate processed images
import glob
import shutil
all_images = glob.glob("dataset_raw/**/*.jpg", recursive=True) + glob.glob("dataset_raw/**/*.png", recursive=True)
for img in all_images:
    shutil.copy(img, config['data']['processed_images'])

In [7]:
# Create YOLO dataset.yaml file
dataset_yaml_content = f"""
path: {os.path.abspath("data/processed")}
train: images
val: images

names:
  0: pothole
"""
with open(config['data']['dataset_yaml'], 'w') as f:
    f.write(dataset_yaml_content.strip())

In [8]:
from ultralytics import YOLO
import time

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [9]:
# 1. Training Run
print("Starting training...")
model = YOLO(config['train']['model_size'])
train_results = model.train(
    data=config['data']['dataset_yaml'],
    epochs=config['train']['epochs'],
    batch=config['train']['batch_size'],
    imgsz=config['train']['img_size'],
    lr0=config['train']['lr0'],
    lrf=config['train']['lrf'],
    weight_decay=config['train']['weight_decay'],
    patience=config['train']['patience'],
    device=0,
    plots=True
)

Starting training...
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=Fal

FileNotFoundError: [34m[1mtrain: [0mError loading data from /content/internship-projects/pothole-detection/data/processed/images
See https://docs.ultralytics.com/datasets for dataset formatting guidance.

In [ ]:
# 2. Validation & Testing Run
print("Running validation...")
val_results = model.val(
    data=config['data']['dataset_yaml'],
    conf=config['evaluation']['conf'],
    iou=config['evaluation']['iou']
)

In [ ]:
# 3. Benchmark Latency & Speed Test
print("Running benchmark latency test...")
print("Running benchmark speed test...")
benchmark_results = model.benchmark(
    data=config['data']['dataset_yaml'],
    imgsz=config['train']['img_size'],
    half=config['export']['half'],
    device=0
)

In [ ]:
# 4. Inference on Sample Images
best_model_path = "runs/detect/train/weights/best.pt"
best_model = YOLO(best_model_path)

sample_images = glob.glob(f"{config['data']['processed_images']}/*.jpg")[:10]
inference_times = []
for img_path in sample_images:
    start_t = time.time()
    _ = best_model(
        img_path,
        conf=config['inference']['conf_threshold'],
        iou=config['inference']['iou_threshold'],
        save=True
    )
    inference_times.append(time.time() - start_t)

avg_latency_ms = (sum(inference_times) / len(inference_times)) * 1000 if inference_times else 0

In [ ]:
# 5. Generate Markdown Report
report_content = f"""# Pothole Detection Model Training & Evaluation Report

## 1. Executive Summary
This report summarizes the training, validation, benchmarking, and inference execution of the YOLOv8 model driven by `config.yaml` on the Pothole Detection dataset using Google Colab Free Tier (NVIDIA T4 GPU).

## 2. Configuration & Training Metrics
* **Model Architecture:** `{cfg['train']['model_size']}`
* **Epochs Trained:** {cfg['train']['epochs']}
* **Batch Size:** {cfg['train']['batch_size']}
* **Image Size:** {cfg['train']['img_size']}x{cfg['train']['img_size']}
* **Compute Cost:** $0.00 (Free Tier)

### Generated Plots & Artifacts
* **Loss & Metric Curves:** `runs/detect/train/results.png`
* **Confusion Matrix:** `runs/detect/train/confusion_matrix.png`
* **Precision-Recall Curves:** `runs/detect/train/PR_curve.png`

## 3. Validation Performance
* **Box mAP50:** {val_results.box.map50:.4f}
* **Box mAP50-95:** {val_results.box.map:.4f}
* **Precision:** {val_results.box.p.mean():.4f}
* **Recall:** {val_results.box.r.mean():.4f}

## 4. Latency & Benchmark Results
* **Average Inference Latency:** {avg_latency_ms:.2f} ms / frame
* **Approximate Throughput:** {1000 / avg_latency_ms:.2f} FPS
* **Weights Path:** `{best_model_path}`

## 5. Sample Inferences
Inference visualizations with confidence filters (`conf={cfg['inference']['conf_threshold']}`) are saved under `runs/detect/predict/`.
"""

with open("Pothole_Detection_Report.md", "w") as f:
    f.write(report_content)

print("Pipeline completed successfully! Report saved to 'Pothole_Detection_Report.md'.")